# A2.6 · The agentic gateway

**Function A — Security Architecture & Platform → The Identity & Non-Human Identity Engineer**  ·  *Security of AI*

---

**Risk.** Secrets end up in agent code because there was nowhere else to put them.

**Control.** Gateway-side credential exchange: virtual keys, JWKS validation, identity mapping.

**This lab.** Move every credential out of the agent and into the gateway.

| | |
|---|---|
| Open-source tooling | agentgateway, kmcp |
| Open-weight models | GLM-4.6 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A2.6"))

An agent gateway is where identity, policy and egress meet. It is the one component that can enforce all three at once, which is exactly why it is worth building deliberately rather than accreting.

In [ ]:
from cybercommons import identity, sandbox

box = sandbox.default_sandbox()

def gateway(token, tool, target=""):
    """Every call: authenticate, authorise by scope, then contain."""
    if token.expired:
        return "DENY  expired token"
    need = {"read_file": "repo:read", "write_file": "repo:write",
            "http_get": "repo:read"}.get(tool)
    if need and need not in token.scopes:
        return f"DENY  scope {need} not in {sorted(token.scopes)}"
    d = box.call(tool, target, approved=(tool in box.tools.require_approval))
    return f"{'ALLOW' if d.allowed else 'DENY '} {d.reason}"

alice = identity.mint("alice")
rev   = identity.exchange(alice, "reviewer-agent", {"repo:read"})
patch = identity.exchange(alice, "patch-agent", {"repo:read", "repo:write"})

for tok, tool, target in [(rev, "read_file", "/work/a.py"),
                          (rev, "write_file", "/work/a.py"),
                          (patch, "write_file", "/work/a.py"),
                          (patch, "http_get", "http://169.254.169.254/"),
                          (patch, "read_file", "/work/../../root/.ssh/id_rsa")]:
    print(f"{tok.actor:16s} {tool:11s} → {gateway(tok, tool, target)}")

Note the last two: `patch-agent` holds every scope it needs and is still refused, because scope is not containment. A gateway that checks only authorisation is half a gateway.

### Expect

`reviewer-agent` is refused `write_file` on scope. `patch-agent` passes the scope check for the same call and succeeds, then is refused the metadata address and the traversal on containment grounds.

### Your turn

The open-source path here is `agentgateway` plus OPA for policy. Which of the three checks above would you put in the gateway, and which in the tool itself? Defend the split.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A2.6.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*